# STEP 15 — 털 가중 샘플러 (1단계 헛알림 직격)

## 왜

헛알림 33.3% 가 가장 큰 약점입니다. 오늘까지 **길을 세 개 지웠습니다**:

| 지운 길 | 근거 |
|---|---|
| 촬영 거리 | `area` AUROC 0.462 · `hair` 와 상관 −0.067 |
| 조명·색 | `sat`/`warm` AUROC 0.53/0.54 |
| **1단계 백본** | STEP 14 판 A — `swinv2` 가 AUROC −0.0018, 흐림 하락 3.0%→10.3% |

남은 건 **학습 데이터를 어떻게 보여주느냐**입니다.

## 무엇을 하나

정상인데 "병원 가보세요" 가 나온 사진 1,234장은 **`hair`(털처럼 가는 선)가
큽니다** — AUROC **0.749**, d 0.80. 그리고 그 값은 견종(0.739)·부위(0.727)·
거리(0.747) 어느 것으로도 설명되지 않는 **사진 한 장의 성질**입니다.

사진 단위 값이니 **샘플러 가중치로 그대로 들어갑니다.** 이미 있는 정상 사진 중
**어려운 것**(잔선 많은 것)을 더 자주 보여줘서 "털 ≠ 병변" 을 배우게 합니다.
**새 데이터도 새 라벨도 필요 없습니다.**

## ★ 총량을 안 바꿉니다 — 이게 설계의 핵심입니다

가중치를 그냥 올리면 정상 사진이 **전체적으로** 더 뽑혀서 클래스 균형이 같이
바뀝니다. 그러면 좋아져도 "털 가중치 덕분" 인지 "정상을 더 봐서" 인지
**못 가릅니다**(교란).

그래서 클래스별 총 가중치를 원래대로 되돌립니다. 바뀌는 건
*정상 안에서 누가 더 뽑히나* 뿐입니다. 아래 셀이 **화면에 찍어서 확인**시켜 줍니다.

⚠️ 1단계는 원래 `balance_strategy="none"` 이라(정상:이상 ≈ 5:5) 클래스 가중치를
안 씁니다. 그래서 이 실험은 **한 가지만** 바꿉니다.

## 판정 기준 (돌리기 **전에** 못 박습니다 — 규칙 2)

* 기준선: `alpha=0` (= 지금과 똑같음). **같은 판에서 같이 돌립니다**
* 채택: 같은 **recall 0.95** 에서 **헛알림(1−precision)이 2%p 이상 감소**
* AUROC 는 참고만 — 우리가 줄이려는 건 헛알림이지 AUROC 가 아닙니다
  (STEP 8 실측: AUROC +0.025 인데 recall 은 −0.015 였습니다)
* 흐림 하락이 **5%p 이상 나빠지면 기각** — 잡음 폭이 ±5%p 입니다

## 왜 여러 alpha 를 한 번에 도나

세기를 모르기 때문입니다. `alpha=0/1/3` 을 같이 돌려 **단조로운지**를 봅니다.
0 → 1 → 3 이 한 방향으로 움직이면 진짜 효과이고, 들쭉날쭉하면 잡음입니다.


In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
NAME   = "deeplearning_test"
# ⚠️ 브랜치를 "main" 으로 **못 박으면 안 됩니다.** 아래 reset --hard 가
#    작업 브랜치를 통째로 덮어써서, 방금 만든 코드가 사라진 채로 몇 시간을
#    돌게 됩니다. 이미 리포 안에서 돌고 있으면 **지금 브랜치를 그대로 씁니다.**
#    바꾸려면 환경변수:  export DOG_SKIN_BRANCH=main
BRANCH = os.environ.get("DOG_SKIN_BRANCH", "")
_cwd   = os.getcwd()
if os.path.basename(_cwd) == NAME and os.path.isdir(os.path.join(_cwd, ".git")):
    DIR = _cwd            # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
    if not BRANCH:
        BRANCH = subprocess.run(["git", "-C", DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                                capture_output=True, text=True).stdout.strip() or "main"
else:
    # ⚠️ Kaggle 을 먼저 봅니다. Kaggle 이미지에도 /content 가 있어서
    #    /content 를 먼저 보면 Kaggle 세션인데 /content 에 clone 합니다.
    BASE = ("/kaggle/working" if os.path.isdir("/kaggle/working")
            else "/content" if os.path.isdir("/content") else _cwd)
    DIR = os.path.join(BASE, NAME)

BRANCH = BRANCH or "main"
if os.path.isdir(os.path.join(DIR, ".git")):
    # 이미 받아둔 경우: 최신으로 강제 동기화 (shallow clone 에서도 안전)
    subprocess.run(["git", "-C", DIR, "fetch", "--depth", "1", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", DIR, "reset", "--hard", f"origin/{BRANCH}"], check=False)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO, DIR], check=True)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", subprocess.run(["git", "-C", DIR, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip())

# 패키지 설치는 **uv 로 통일**합니다 (pip 보다 훨씬 빠릅니다).
# ⚠️ Colab/Kaggle 이미지에는 uv 가 없어서, uv 자체만 pip 로 한 번 받습니다.
#    --system = 가상환경을 새로 만들지 않고 이미 있는 파이썬에 그대로 설치.
#    (torch/numpy/pandas 는 이미 깔려 있으므로 여기서 안 건드립니다)
# albumentations 는 import 할 때마다 PyPI 에 버전 확인 요청을 보냅니다.
# Kaggle 은 외부 네트워크가 막혀 있어 타임아웃(2초)만 기다리다 끝납니다 — 꺼둡니다.
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

_PKGS = ["timm", "imagehash", "pyarrow", "grad-cam", "albumentations"]
_ok = False
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"],
                  check=False).returncode == 0:
    _ok = subprocess.run([sys.executable, "-m", "uv", "pip", "install", "-q",
                          "--system", *_PKGS], check=False).returncode == 0
if not _ok:
    print("[env] uv 로 설치하지 못해 pip 으로 대체합니다")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_PKGS], check=False)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

MY_NOTEBOOK_VERSION = "2026-08-25.1"   # ★ 이 셀(=이 .ipynb)의 버전

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

# 환경 판정이 이상하면(예: Kaggle 인데 colab 이라고 나오면) 근거를 봅니다
if E.env != "local":
    env.diagnose()

# ⚠️ 노트북 셀은 git pull 로 갱신되지 않습니다 (src/ 만 최신이 됩니다).
#    낡은 .ipynb 를 몇 시간 돌리고 나서 알게 되면 늦으므로 지금 확인합니다.
from src.config import NOTEBOOK_VERSION as _repo_nb
if MY_NOTEBOOK_VERSION != _repo_nb:
    print("\n" + "!" * 62)
    print(f"⚠️ 이 노트북이 낡았습니다 — 내 셀 {MY_NOTEBOOK_VERSION} / 리포 {_repo_nb}")
    print("   src/ 는 최신이지만 **셀 내용은 예전 것**입니다.")
    print("   GitHub 에서 notebooks/*.ipynb 를 다시 받아 Import 하세요:")
    print("   Kaggle → File → Import Notebook / Colab → 파일 → 노트 업로드")
    print("!" * 62 + "\n")
else:
    print(f"[nb] 노트북 최신 ({_repo_nb})")


---
## 1. 데이터 붙이기 + 사전 검증

In [ ]:
# Drive 마운트는 **진짜 Colab VM** 에서만 시도합니다.
# ⚠️ Kaggle 에도 google.colab 패키지와 /content 가 있어서, 환경 판정을 잘못하면
#    Kaggle 에서 drive.mount() 를 부르고 NotImplementedError 로 죽습니다.
if env.can_mount_drive():
    env.mount_drive()
else:
    print(f"[env] {E.env} — Drive 마운트 없이 진행합니다")

# 전처리 결과를 붙입니다. 두 가지 형태를 다 받습니다:
#   · Colab  : Drive 의 dogskin_prepared.zip → 로컬 디스크로 해제
#   · Kaggle : /kaggle/input/<데이터셋>/crops,manifests → 링크만 연결
#              (Kaggle 은 업로드한 zip 을 알아서 풀어둡니다. 복사하면 20GB 제한에 걸려요)
env.load_prepared()          # 경로를 직접 주려면: env.load_prepared("/kaggle/input/dogskin-prepared")

# ── 다른 환경에서 학습한 체크포인트 가져오기 (Colab → Kaggle 이주) ──────
#    Colab 에서 이미 학습을 끝냈다면, Drive 의 dogskin_work/checkpoints 를
#    Kaggle 데이터셋으로 올린 뒤 그 경로를 여기에 주세요.
#    가져온 실험은 '완료' 로 인식되어 학습 셀이 ⏭️ 로 건너뜁니다.
#
# train.import_checkpoints("/kaggle/input/dogskin-ckpt")

# 세션이 끊겨도 남는 저장소 확인
_persist = env.persist_root()
if _persist is None:
    print("\n🚨 세션 밖 저장소가 없습니다 — 지금 학습하면 끊길 때 체크포인트가 사라집니다.")
    print("   위 셀에서 Drive 마운트가 됐는지 확인하세요 (env.mount_drive()).")
else:
    print(f"\n✅ 중단 대비 저장소: {_persist}")
    if E.env == "kaggle":
        print("   ⚠️ Kaggle 은 세션이 끝나면 /kaggle/working 이 사라질 수 있습니다.")
        print("      · 짧게 확인만 할 때  : 그냥 진행 (세션 안에서는 이어받기가 됩니다)")
        print("      · 긴 학습을 돌릴 때  : 우측 상단 [Save Version] →")
        print("                             'Save & Run All (Commit)' 로 돌리세요.")
        print("                             브라우저를 닫아도 끝까지 돌고, 출력이 보존됩니다.")
        print("      · 설정에 Persistence 항목이 보이면 'Files' 로 켜두면 더 안전합니다")
    else:
        print("   매 에폭 체크포인트를 여기로 복사합니다. 세션이 끊기면 노트북을 처음부터")
        print("   다시 돌리세요 — 끝난 학습은 건너뛰고 끊긴 학습만 이어서 합니다.")

In [ ]:
import torch
from src import (labels, split, crop, data, models, train, evaluate,
                 stages, experiments, robust, texture)
from src.config import CLASSES_STAGE1

env.require_gpu()
DEV = "cuda" if torch.cuda.is_available() else "cpu"

EPOCHS      = 12           # 서브셋 스윕용 (03b·03d·03f 와 동일)
SUBSET      = 0.55         # 학습셋만. 검증셋은 그대로
STAGE1_CROP = "f320"       # STEP 9-A 확정
MODEL, IMG_SIZE = "effnetv2_s", 384      # STEP 14 판 A 에서 유지 확정
AUG = "photometric"        # STEP 6 에서 확정

# ★ 기준선(0)을 **같은 판에** 넣습니다. 04 에서 같은 설정 resnet50 이
#   판 A 0.4862 / 판 B 0.5024 (+0.0162) 로 달랐습니다 — 판을 넘으면 못 믿습니다.
ALPHAS = [0.0, 1.0, 3.0]
BASE_ALPHA = 0.0

# 사전 등록 판정 문턱 (규칙 2 — 결과 보고 못 바꿉니다)
FA_DROP_ACCEPT = 0.02      # 헛알림 2%p 이상 감소해야 채택
BLUR_WORSE_REJECT = 0.05   # 흐림 하락이 5%p 이상 나빠지면 기각

# STEP 14 판 A 의 같은 설정 (참고용 — **판이 달라 절대값 비교는 금지**)
REF_STEP14 = {"auroc": 0.9465, "blur_drop": 0.0304, "precision": 0.797}

df = labels.load(env.work_root()/"manifests"/"manifest_final.parquet")

# ── 학습 전에 전부 확인합니다 ────────────────────────────────────
have = crop.available_tags()
print(f"사용 가능한 크롭 태그: {have}")
if STAGE1_CROP not in have:
    raise SystemExit(
        f"❌ 크롭 '{STAGE1_CROP}' 이 없습니다. 붙어 있는 것: {have}\n"
        f"   1단계만 도는 노트북이라 f320 만 있으면 됩니다.")

d = crop.switch_tag(df, STAGE1_CROP)
view = stages.to_stage1(d)
split.verify(view, fold=0, strict=True)
tr, va = split.get_fold(view, 0)

N_TRAIN = int(len(tr) * SUBSET)
print(f"\n1단계 뷰 {len(view):,}행  ·  train {len(tr):,} → {N_TRAIN:,}({SUBSET:.0%})"
      f"  ·  val {len(va):,}")
print(f"조건 {len(ALPHAS)}개: alpha {ALPHAS}  (기준선 {BASE_ALPHA:g})")
if BASE_ALPHA not in ALPHAS:
    raise SystemExit(f"❌ 기준선 alpha={BASE_ALPHA} 가 ALPHAS 에 없습니다 — 비교 불가.")


---
## 2. 시작 전에 — 몇 시간 걸릴지 먼저 잽니다

합성 텐서로 GPU 속도만 재므로 **백본당 20초** 안쪽입니다. 학습은 아직 시작 안 합니다.

> 이번 프로젝트에서 "몇 시간 걸릴지 모르고 돌렸다가 뒤통수" 를 여러 번 맞았습니다.
> 여기서 총 예상 시간을 보고 **너무 길면 그만두거나 서브셋을 줄이세요.**

⚠️ GPU 속도만 잰 **하한**입니다. 데이터 로딩이 병목이면 실제는 더 걸립니다
(실측: 384px 에서 GPU 112 img/s 상한, 로더 90 img/s).

In [ ]:
# ── hair 를 **미리** 재둡니다 ──────────────────────────────────
# 학습 중에 재면 첫 에폭이 느려지고, 세 번 도는 동안 같은 값을 세 번 잽니다.
# 캐시에 남겨두면 이번 실행에서 한 번만 재고 다음부터는 공짜입니다.
_cache = env.work_root()/"reports"/"hair_index.parquet"
_norm = tr[tr["label"] == stages.NORMAL_LABEL]
print(f"[hair] 정상 사진 {len(_norm):,}장을 잽니다 (캐시: {_cache.name})")
_h = texture.hair_index(_norm["crop_path"].tolist(), cache=_cache)
import numpy as np
print(f"[hair] 분위 {np.round(np.quantile(_h, [0,.25,.5,.75,1]), 4).tolist()}")

est = experiments.estimate_runtime(
    [MODEL], img_size=IMG_SIZE, n_train=N_TRAIN, epochs=EPOCHS,
    n_conditions=len(ALPHAS))

QUOTA_H = 4.0        # ← 쓸 수 있는 GPU 시간 (런팟이면 예산/시간당요금)
_need = est["total_hours"] * 1.45 + 0.3      # 로딩 병목(STEP 6·14 실측) + 교란 검사
print(f"\n  쓸 수 있는 시간 {QUOTA_H:.1f}h  vs  필요 추정 {_need:.1f}h")
if _need > QUOTA_H:
    print("  ❌ 모자랍니다.")
    print("     ① ALPHAS 를 [0.0, 1.0] 둘로 (기준선은 절대 빼지 마세요)")
    print("     ② SUBSET 0.55 → 0.35")
    raise SystemExit("시간 부족 — 위대로 줄이고 이 셀을 다시 도세요.")
print("  ✅ 들어갑니다.")


---
## 3. 2×2 학습

⚠️ 여기서부터 오래 걸립니다. 위 예상 시간을 보고 진행하세요.
세션이 끊겨도 `train.fit(resume=True)` 가 마지막 에폭부터 이어받습니다.

In [ ]:
# ── 스윕 ────────────────────────────────────────────────────
# ★ 이어받기: 끝난 조건은 train.fit 이 건너뜁니다 (이름에 alpha 가 들어갑니다).
import gc

runs = []
for _a in ALPHAS:
    try:
        runs.append(experiments.train_and_measure(
            view, stage=1, img_size=IMG_SIZE, crop_tag=STAGE1_CROP,
            device=DEV, epochs=EPOCHS, model_name=MODEL, aug=AUG,
            subset_frac=SUBSET,
            balance="hair_weighted" if _a > 0 else "none",
            hair_alpha=_a,
            measure_robust=False, measure_blur=True, n_robust=2000))
    except torch.cuda.OutOfMemoryError:
        print(f"❌ alpha={_a}: VRAM 부족")
    except Exception as exc:                                    # noqa: BLE001
        print(f"❌ alpha={_a}: {type(exc).__name__}: {exc}")
    finally:
        gc.collect()
        if DEV == "cuda":
            torch.cuda.empty_cache()

if not runs:
    raise SystemExit("❌ 성공한 실행이 0개입니다.")
if not any(r["hair_alpha"] == BASE_ALPHA for r in runs):
    raise SystemExit(f"❌ 기준선(alpha={BASE_ALPHA})이 실패했습니다 — 비교할 기준이 없습니다.")


---
## 4. 판정

기준은 `experiments.stage1_report()` 에 있습니다 — **실험 전에** 정해뒀고
노트북 셀이 아니라 `src/` 에 있어서 `git pull` 로 갱신됩니다 (규칙 2·3).

In [ ]:
# ── 판정 ────────────────────────────────────────────────────
import json

base = next(r for r in runs if r["hair_alpha"] == BASE_ALPHA)

def fa(r):
    """헛알림 = 이상이라 한 것 중 실제로는 정상인 비율 (= 1 − precision).
    recall 은 0.95 로 고정돼 있으므로 이 값만 비교하면 됩니다."""
    return 1.0 - r["precision"]

print("\n" + "=" * 74)
print(" 털 가중 샘플러 — 헛알림이 줄었나")
print("=" * 74)
print(f"  {'alpha':>6}{'AUROC':>10}{'헛알림':>10}{'Δ헛알림':>10}"
      f"{'흐림하락':>10}{'Δ흐림':>9}{'분':>6}")
print("  " + "─" * 66)
for r in sorted(runs, key=lambda x: x["hair_alpha"]):
    d_fa = fa(r) - fa(base)
    d_bl = (r.get("blur_drop") or 0) - (base.get("blur_drop") or 0)
    tag = "  ← 기준선" if r["hair_alpha"] == BASE_ALPHA else ""
    print(f"  {r['hair_alpha']:>6.1f}{r['auroc']:>10.4f}{fa(r):>10.1%}"
          f"{d_fa:>+10.1%}{(r.get('blur_drop') or 0):>10.1%}{d_bl:>+9.1%}"
          f"{r['minutes']:>6.0f}{tag}")

print(f"\n  판정 문턱 (돌리기 전에 정함): 헛알림 −{FA_DROP_ACCEPT:.0%} 이상 "
      f"· 흐림 하락 +{BLUR_WORSE_REJECT:.0%} 이상 나빠지면 기각")

cand = []
for r in runs:
    if r["hair_alpha"] == BASE_ALPHA:
        continue
    d_fa, d_bl = fa(r) - fa(base), (r.get("blur_drop") or 0) - (base.get("blur_drop") or 0)
    if d_bl >= BLUR_WORSE_REJECT:
        print(f"  ❌ alpha={r['hair_alpha']:g} — 흐림 하락이 {d_bl:+.1%} 나빠져 기각")
    elif d_fa <= -FA_DROP_ACCEPT:
        print(f"  ✅ alpha={r['hair_alpha']:g} — 헛알림 {d_fa:+.1%}")
        cand.append((r["hair_alpha"], d_fa, r))
    else:
        print(f"  ➖ alpha={r['hair_alpha']:g} — 헛알림 {d_fa:+.1%} (문턱 미달)")

# 단조성 — 0→1→3 이 한 방향이면 진짜 효과, 들쭉날쭉하면 잡음
_s = sorted(runs, key=lambda x: x["hair_alpha"])
_fas = [fa(r) for r in _s]
mono = all(b <= a + 1e-9 for a, b in zip(_fas, _fas[1:])) or \
       all(b >= a - 1e-9 for a, b in zip(_fas, _fas[1:]))
print(f"\n  헛알림이 alpha 에 대해 단조인가: {'예' if mono else '아니오'} "
      f"{[f'{v:.1%}' for v in _fas]}")
if not mono:
    print("     ⚠️ 들쭉날쭉합니다 — 효과보다 잡음일 수 있습니다. 채택을 보류하세요.")

if cand:
    best = min(cand, key=lambda t: t[1])
    print(f"\n  ★ 채택: alpha={best[0]:g}  (헛알림 {best[1]:+.1%})")
    print(f"     → 06 재학습에 balance='hair_weighted', hair_alpha={best[0]:g}")
else:
    best = None
    print("\n  ★ 채택 없음 — 털 가중치로는 헛알림이 안 줄어듭니다.")
    print("     이것도 결론입니다. 남은 건 앱 촬영 가이드(문구)뿐입니다.")

print("\n  ⚠️ STEP 14 판 A 와 **절대값을 비교하지 마세요** (판이 다릅니다).")
print(f"     참고: 그때 같은 설정이 AUROC {REF_STEP14['auroc']:.4f} · "
      f"헛알림 {1 - REF_STEP14['precision']:.1%} 였습니다.")
print("=" * 74)

W = env.work_root(); (W/"reports").mkdir(parents=True, exist_ok=True)
keep = ("stage", "model_name", "aug", "balance", "hair_alpha", "img_size",
        "crop_tag", "exp_name", "epochs", "batch_size", "minutes", "best_epoch",
        "n_epochs", "converged", "subset_frac", "n_train", "auroc", "threshold",
        "precision", "blur_drop", "blur_worst", "blur_worst_at")
(W/"reports"/"step15_hair_sampler.json").write_text(json.dumps({
    "step": "STEP15_털가중샘플러",
    "model": MODEL, "img_size": IMG_SIZE, "crop": STAGE1_CROP, "aug": AUG,
    "epochs": EPOCHS, "subset_frac": SUBSET, "alphas": ALPHAS,
    "thresholds": {"fa_drop_accept": FA_DROP_ACCEPT,
                   "blur_worse_reject": BLUR_WORSE_REJECT},
    "ref_step14_판A": REF_STEP14,
    "monotonic": bool(mono),
    "runs": [{k: r[k] for k in keep if k in r} for r in runs],
    "picked": (best[0] if best else None),
}, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\n저장: {W/'reports'/'step15_hair_sampler.json'}")

train.export_release(
    exps=[r["exp_name"] for r in runs],
    meta={"실험": "STEP 15 털 가중 샘플러", "크롭": STAGE1_CROP,
          "백본": MODEL, "해상도": IMG_SIZE, "서브셋": f"{SUBSET:.0%}",
          "에폭": EPOCHS, "alpha": str(ALPHAS),
          "채택": str(best[0] if best else "없음")},
    files={"reports/step15_hair_sampler.json": json.loads(
        (W/"reports"/"step15_hair_sampler.json").read_text(encoding="utf-8"))},
)


---
## 5. 다음

**이 노트북은 후보를 고르는 것까지입니다.** 서브셋 55% · 12에폭이라 절대값은
풀 학습과 다릅니다.

| 결과 | 다음 |
|---|---|
| 어느 축이든 채택됨 | 그 조합으로 **03 을 풀 데이터 재실행** → holdout 을 다시 엽니다 |
| 둘 다 잡음 안 | 두 축 모두 닫고 `f320` 재크롭(구도 불일치 가설)으로 |

⚠️ **holdout 은 아직 안 봤습니다.** 풀 학습 뒤에 한 번만 엽니다.
지금 holdout 을 보고 조합을 고르면 그 숫자는 더 이상 정직하지 않습니다.

⚠️ Kaggle 이면 **[Save Version] → Save & Run All (Commit)** 으로 돌리고,
끝나면 `release` 폴더를 **New Dataset (Private)** 으로 만들어 두세요.
`READ_ME_FIRST.txt` 에 어떤 실험인지 적혀 있습니다.